In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, f1_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer, plot_visualizer_classification
import plotly.graph_objects as go
from sklearn.model_selection import StratifiedKFold
from preprocessing import get_features_and_target_classification
from tabpfn import TabPFNClassifier, TabPFNRegressor
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/test_data.csv")
#sc = StandardScaler()

target_column_class = "Category" 

x_train, y_train  = get_features_and_target(train_df, target_column_class)
x_dev, y_dev = get_features_and_target(dev_df, target_column_class)

#x_train = sc.fit_transform(X=x_train)
#x_dev = sc.transform(x_dev)

# Defining Model

In [11]:
#model_name_class = 'XGBoostClassifier'
model_name_class = 'RandomForestClassifier'
#model_name_class = 'TabPFNClassifier'

model_name_reg = 'XGBoost'
#model_name_reg = 'RandomForest'
#model_name_reg = 'TabPFN'

# Add Physical Columns Interfacial_Failure and Pullout_Failure

In [5]:
def compute_interfacial_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = 1 * (np.pi/4) * (4 * np.sqrt(t))**2 * (0.7 * 365) 
     return np.round(f_pull, 1) 

def compute_pullout_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     x = df[['Thickness A (mm)', 'Thickness B (mm)']].sum(axis=1)
     # x can be approximated to metal sheet thickness. Change 2*t either to t to use the thinner 
     # metal sheet or 2*x to test if the sum of both metal sheets give beter results
     f_pull = np.pi * ((4 * np.sqrt(t)) + 2*t)*t*365 
     return np.round(f_pull, 1) 

# Fit Model

In [12]:
# Target value
if model_name_class == 'RandomForestClassifier':

        
        params = {
            'n_estimators': 27,
            'max_depth': 7,
            'min_samples_split': 5,
            'min_samples_leaf': 2,
            'max_features': 'log2',
        }

        classifier = RandomForestClassifier(**params)
        classifier.fit(x_train, y_train)

        preds = classifier.predict(x_dev)



# Add Pullforce as Target

In [13]:
def compute_pullforces(x, y, predictions_class):
    pullforces = []

    for sample_id, pred in zip(y.index, predictions_class):
        row_x_df = x.loc[[sample_id]]  # 1-row DataFrame

        if pred == "Bad":
            value = compute_interfacial_failure(row_x_df).iloc[0]
        else:
            value = compute_pullout_failure(row_x_df).iloc[0]

        pullforces.append(value)

    return np.array(pullforces)


In [14]:
predictions_class_train = classifier.predict(x_train)
x_train["Physical Inference"] = compute_pullforces(x_train, y_train, predictions_class_train)


predictions_class_dev = classifier.predict(x_dev)
x_dev["Physical Inference"] = compute_pullforces(x_dev, y_dev, predictions_class_dev)


In [15]:
y_train_Regressor = train_df.groupby("Sample ID")['PullTest (N)'].first()
y_dev_Regressor = dev_df.groupby("Sample ID")['PullTest (N)'].first()

# Fit 2nd Model

In [17]:
if model_name_reg == 'XGBoost':

        # Convert the data into DMatrix format
        dtrain = xgb.DMatrix(x_train, label=y_train_Regressor)
        dtest = xgb.DMatrix(x_dev, label=y_dev_Regressor)

        # Set the parameters for the XGBoost model
        params = {
            'objective': 'reg:squarederror',
            'max_depth': 1,
            'eta': 0.57,
            'eval_metric': 'rmse',
        }

        # Train the model
        num_boost_round = 20
        bst = xgb.train(params, dtrain, num_boost_round)

        # Make predictions
        predictions_regression_dev = bst.predict(dtest)


# Check Validation Data

In [20]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev_Regressor,
    pred_vals=predictions_regression_dev,
    categories=categories,
    title=f"Validation Samples: True vs Prediction ({model_name_reg}) by Category"
)

# Check Validation Loss and R2

In [19]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev_Regressor, predictions_regression_dev)
rmse = np.sqrt(mean_squared_error(y_dev_Regressor, predictions_regression_dev))
R2   = r2_score(y_dev_Regressor, predictions_regression_dev)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")

MAE:  140.09
RMSE: 204.30
R2: 0.76
